# Twelve starts with and without a stationarity check at iteration 200

Paired experiment on the original 50 AR(3) coefficient-recovery paths from `ar3_minimum_width_k4.ipynb`.

Both modes use **all 12 original initial directions**, **max_it=2000**, **improv_patience=200**, and the same normalized geometric step schedule.

- Ordinary mode runs each attempt to the usual stopping point.
- Triggered mode checks the coefficients of the **best-width incumbent after 200 evaluated iterations**. It aborts that attempt if the companion spectral radius is at least one or the coefficients are undefined. Otherwise it continues the unchanged trajectory and schedule.
- Aborted attempts cannot win. The final result is the smallest-width estimate among completed attempts, without a final stationarity filter or selection among stationary intermediate iterates.
- All 12 attempts are always used, even after one succeeds. An initial attempt plus 11 new starts is the same bank of 12 starts used by the original experiment.

The parameter validity check uses the companion spectral radius < 1; equivalently the AR lag-polynomial roots lie outside the unit circle. Sources: [statsmodels stationarity](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima_process.ArmaProcess.isstationary.html), [reciprocal-root equivalence, Notes](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.tools.is_invertible.html#notes).

Iteration counts follow the existing optimizer: the initial state is evaluated iteration 1. The check reads state index 199, before another update. The true coefficients never enter decisions.


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
PROJECT_ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents)
                  if (p/'Code/Code_AR_processes/ar_tools/descent.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
from output.analysis.ar3_early_abort_200_12starts.run_experiment import main as run_experiment
from output.analysis.ar3_early_abort_200_12starts.verify_and_report import plot_and_report
DATA_DIR=PROJECT_ROOT/'Code/Code_AR_processes/outputs/csv/ar3_early_abort_200_12starts'


In [2]:
summary=run_experiment()


Completed 1/50 (0.0s)
Completed 5/50 (0.0s)
Completed 10/50 (0.0s)
Completed 15/50 (0.0s)
Completed 20/50 (0.0s)
Completed 25/50 (0.0s)
Completed 30/50 (0.0s)
Completed 35/50 (0.0s)
Completed 40/50 (0.0s)
Completed 45/50 (0.0s)
Completed 50/50 (0.0s)
                              method  processes  stationary_estimates  failed_no_survivor  mean_coefficient_mse  median_coefficient_mse  maximum_coefficient_mse  mse_above_1  total_attempts  total_aborted  total_evaluated_states
           12 starts, no early abort         50                    46                   0            218.295726                0.000015              7680.880243            4             600              0                  600005
12 starts, stationarity abort at 200         50                    49                   0              0.006610                0.000015                 0.269066            0             600            179                  432283
{
  "improved": 6,
  "worsened": 4,
  "tied": 40,
  "iteration

In [3]:
image_dir=plot_and_report()
print(json.dumps(json.loads((DATA_DIR/"validation.json").read_text()),indent=2))


{
  "processes": 50,
  "policy_results_checked": 100,
  "attempts_per_policy_checked": 600,
  "all_600_prefixes_match_public_optimizer": true,
  "checkpoint_roots_independently_checked": true,
  "all_1200_candidate_widths_and_coefficients_checked": true,
  "all_1200_trace_stopping_counts_checked": true,
  "all_final_selections_and_iteration_costs_checked": true,
  "maximum_candidate_width_error": 0.0,
  "baseline_evaluations": 600005,
  "policy_evaluations": 432283,
  "aborted_runs_that_would_recover": 47
}


The trigger aborts 179 of 600 attempts and saves **27.95% of evaluated iterations**. Coefficient MSE improves on 6 paths, worsens on 4, and is tied on 40. The four ordinary-mode MSE-above-1 cases are avoided; the median MSE is unchanged.

Of the aborted attempts, **47 would eventually recover stationarity**. Conversely, one selected result passes the checkpoint but later becomes nonstationary. The policy therefore returns 49/50 stationary estimates, compared with 46/50 without the trigger. The 200-iteration check is a heuristic, not a certificate that a direction cannot recover or that a surviving trajectory remains stationary.

![Accuracy and evaluated iterations](outputs/images/ar3_early_abort_200_12starts/01_comparison.png)

![Checkpoint examples](outputs/images/ar3_early_abort_200_12starts/02_checkpoint_examples.png)

Dashed continuation after an abort shows what would happen in the ordinary run. It is only used for retrospective analysis, never for the checkpoint decision.

[Full findings](outputs/csv/ar3_early_abort_200_12starts/findings.md) · [All policy results](outputs/csv/ar3_early_abort_200_12starts/per_process_results.csv) · [Checkpoint diagnostics](outputs/csv/ar3_early_abort_200_12starts/per_start_diagnostics.csv) · [Validation](outputs/csv/ar3_early_abort_200_12starts/validation.json) · [Comparison PDF](outputs/pdfs/ar3_early_abort_200_12starts/01_comparison.pdf) · [Checkpoint PDF](outputs/pdfs/ar3_early_abort_200_12starts/02_checkpoint_examples.pdf)


## Detailed iteration examples

Each curve tracks the companion spectral radius of the best-width incumbent's recovered coefficients. The vertical purple line marks the decision at iteration 200. The dotted horizontal line is the stationarity boundary at 1; the black dashed line is the true parameter radius.

Regenerate these figures from the saved trajectories, without rerunning or changing the descent fits:


In [4]:
from output.analysis.ar3_early_abort_200_12starts.plot_counterexamples import main as plot_counterexamples
detailed_examples = plot_counterexamples()


[
  {
    "replication": 8,
    "start": 11,
    "aborted": true,
    "checkpoint_iteration": 200,
    "checkpoint_radius": 1.1980245396259122,
    "final_iteration": 2000,
    "final_radius": 0.9551279989442084,
    "true_radius": 0.9546697236176747,
    "first_stationary_iteration_after_checkpoint": 371,
    "first_nonstationary_iteration_after_checkpoint": 201,
    "checkpoint_coefficient_mse": 1.14839355951714,
    "final_coefficient_mse": 0.0003893505676345705,
    "png": "/Users/iljasukhanov/MASTERARBEIT/Code/Code_AR_processes/outputs/images/ar3_early_abort_200_12starts/03_aborted_run_that_recovers.png",
    "pdf": "/Users/iljasukhanov/MASTERARBEIT/Code/Code_AR_processes/outputs/pdfs/ar3_early_abort_200_12starts/03_aborted_run_that_recovers.pdf"
  },
  {
    "replication": 43,
    "start": 2,
    "aborted": false,
    "checkpoint_iteration": 200,
    "checkpoint_radius": 0.9316812604659467,
    "final_iteration": 494,
    "final_radius": 1.0082980892621507,
    "true_radius": 0.9

**Aborted but would recover: process 8, start 11.** Radius 1.1980 at 200 triggers abortion. Without that abortion, the estimate becomes stationary at iteration 371 and remains stationary through iteration 2000, ending at radius 0.9551. The dashed blue portion is this continuation without abortion.

![Aborted but later recovered](outputs/images/ar3_early_abort_200_12starts/03_aborted_run_that_recovers.png)

[PDF](outputs/pdfs/ar3_early_abort_200_12starts/03_aborted_run_that_recovers.pdf)

**Passed but later became nonstationary: process 43, start 2.** Radius 0.9317 at 200 passes. At iteration 256 it becomes nonstationary and stays so until the final evaluated iteration 494, ending at radius 1.0083. This is the selected run for process 43 under the early-abort method.

![Passed but later nonstationary](outputs/images/ar3_early_abort_200_12starts/04_passed_run_that_becomes_nonstationary.png)

[PDF](outputs/pdfs/ar3_early_abort_200_12starts/04_passed_run_that_becomes_nonstationary.pdf)

[All plotted iterations](outputs/csv/ar3_early_abort_200_12starts/checkpoint_counterexamples_history.csv) · [Exact event values](outputs/csv/ar3_early_abort_200_12starts/checkpoint_counterexamples.json) · [Detailed report](outputs/csv/ar3_early_abort_200_12starts/checkpoint_counterexamples.md)
